In [ ]:
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, GRU, Dense, TimeDistributed
from tensorflow.keras.losses import BinaryCrossentropy, MeanSquaredError
from tensorflow.keras.optimizers import Adam
from sklearn.preprocessing import MinMaxScaler
import warnings

warnings.filterwarnings("ignore")

# --- 1. Configuration ---
INPUT_FILE = '../../data/data_with_features.csv'
OUTPUT_FILE = '../../data/syntheticData/synthetic_data_gan2.csv'
TARGET_COLUMN = 'cumulative_mite_count'
GROUP_COLUMN = 'treatment_id'

# Model Parameters
HIDDEN_UNITS = 32
LATENT_DIM = 24
NUM_LAYERS = 3
EPOCHS = 5000
BATCH_SIZE = 128

# --- 2. Data Preparation (Unchanged) ---
print("--- ⚙️ Preparing Data for TimeGAN ---")
df = pd.read_csv(INPUT_FILE)
grouped = df.groupby(GROUP_COLUMN)
original_series_list = [g[TARGET_COLUMN].values.astype(float) for _, g in grouped]
original_lengths = [len(s) for s in original_series_list]
max_len = max(original_lengths)
print(f"   Found {len(original_series_list)} time series groups.")
print(f"   Maximum series length is {max_len}.")
padded_series = np.array([np.pad(s, (0, max_len - len(s)), 'constant', constant_values=0) for s in original_series_list])
scaler = MinMaxScaler()
scaled_data = scaler.fit_transform(padded_series)
data_for_gan = scaled_data.reshape(len(scaled_data), max_len, 1)
print(f"   Data prepared with shape: {data_for_gan.shape}\n")

# --- 3. TimeGAN Model Components ---
print("--- 🏗️ Building TimeGAN Components ---")

# Helper function to create GRU stacks
def make_gru_network(units, layers, return_sequences=True, name=None):
    model = tf.keras.Sequential(name=name)
    for _ in range(layers):
        model.add(GRU(units, return_sequences=return_sequences))
    return model

# a. Autoencoder Components
encoder_input = Input(shape=(max_len, 1))
encoder_net = make_gru_network(HIDDEN_UNITS, NUM_LAYERS, name='EncoderGRU')(encoder_input)
encoder = Model(encoder_input, encoder_net, name='Encoder')

decoder_input = Input(shape=(max_len, HIDDEN_UNITS))
decoder_net = make_gru_network(HIDDEN_UNITS, NUM_LAYERS, name='DecoderGRU')(decoder_input)
decoder_output = TimeDistributed(Dense(1), name='DecoderOutput')(decoder_net)
decoder = Model(decoder_input, decoder_output, name='Decoder')

autoencoder = Model(encoder_input, decoder(encoder(encoder_input)), name='Autoencoder')

# b. Adversarial Components
generator_input = Input(shape=(max_len, LATENT_DIM))
generator_net = make_gru_network(HIDDEN_UNITS, NUM_LAYERS, name='GeneratorGRU')(generator_input)
generator = Model(generator_input, generator_net, name='Generator')

discriminator_input = Input(shape=(max_len, HIDDEN_UNITS))
disc_net = GRU(HIDDEN_UNITS, name='DiscriminatorGRU')(discriminator_input)
disc_output = Dense(1, activation='sigmoid', name='DiscriminatorOutput')(disc_net)
discriminator = Model(discriminator_input, disc_output, name='Discriminator')

# --- 4. Compile Models (CORRECTED ORDER) ---
print("--- 🛠️ Compiling Models ---")

# Compile Autoencoder
autoencoder.compile(optimizer=Adam(), loss=MeanSquaredError())

# Compile Discriminator
discriminator.compile(optimizer=Adam(), loss=BinaryCrossentropy())
discriminator.trainable = False # Freeze discriminator for the combined GAN model

# Compile Combined GAN Model (Generator + Frozen Discriminator)
gan_input = generator_input
gan_output = discriminator(generator(gan_input))
gan = Model(gan_input, gan_output, name="GAN")
gan.compile(optimizer=Adam(), loss=BinaryCrossentropy())

print("   All models compiled successfully.\n")

# --- 5. Training Loop ---
print(f"--- 🏋️ Training TimeGAN for {EPOCHS} epochs ---")

# Phase 1: Train Autoencoder
print("   Phase 1/3: Training Autoencoder...")
autoencoder.fit(data_for_gan, data_for_gan, batch_size=BATCH_SIZE, epochs=EPOCHS, verbose=0)
print("   Autoencoder training complete.")

# Phase 2 & 3: Joint Adversarial Training
print("\n   Phases 2 & 3: Jointly Training Generator and Discriminator...")
for epoch in range(EPOCHS):
    # Select a random batch of real data
    idx = np.random.randint(0, data_for_gan.shape[0], BATCH_SIZE)
    real_batch = data_for_gan[idx]
    
    # Generate a batch of synthetic data
    random_noise = tf.random.normal(shape=(BATCH_SIZE, max_len, LATENT_DIM))
    synthetic_latent = generator.predict(random_noise, verbose=0)
    
    # Get the real latent codes
    real_latent = encoder.predict(real_batch, verbose=0)
    
    # Train the discriminator
    discriminator.trainable = True
    d_loss_real = discriminator.train_on_batch(real_latent, np.ones(BATCH_SIZE))
    d_loss_fake = discriminator.train_on_batch(synthetic_latent, np.zeros(BATCH_SIZE))
    d_loss = 0.5 * (d_loss_real + d_loss_fake)
    discriminator.trainable = False
    
    # Train the generator
    random_noise = tf.random.normal(shape=(BATCH_SIZE, max_len, LATENT_DIM))
    g_loss = gan.train_on_batch(random_noise, np.ones(BATCH_SIZE))
    
    if epoch % 100 == 0:
        print(f"   Epoch {epoch}, Disc Loss: {d_loss:.4f}, Gen Loss: {g_loss:.4f}")
print("   Joint training complete.\n")

# --- 6. Generate Synthetic Data ---
print("--- ✨ Generating New Synthetic Data ---")
num_samples_to_generate = len(original_series_list)
random_Z_final = tf.random.normal(shape=(num_samples_to_generate, max_len, LATENT_DIM))

generated_latent = generator.predict(random_Z_final, verbose=0)
generated_scaled = decoder.predict(generated_latent, verbose=0)
generated_padded = scaler.inverse_transform(generated_scaled.reshape(-1, max_len))

generated_final = []
for i, length in enumerate(original_lengths):
    series = generated_padded[i, :length]
    series = np.maximum(0, series)
    series = np.maximum.accumulate(series)
    generated_final.append(series)
print("   Synthetic data generated and post-processed.\n")

# --- 7. Save the New Dataset ---
print(f"--- 💾 Saving to {OUTPUT_FILE} ---")
synthetic_df_list = []
original_groups = [g for _, g in grouped]
for i, group_df in enumerate(original_groups):
    temp_df = group_df.copy()
    temp_df[TARGET_COLUMN] = generated_final[i]
    synthetic_df_list.append(temp_df)

final_synthetic_df = pd.concat(synthetic_df_list, ignore_index=True)
final_synthetic_df.to_csv(OUTPUT_FILE, index=False)
print(f"   Successfully saved synthetic data. Final shape: {final_synthetic_df.shape}")
print("\n--- ✅ GAN Process Complete ---")

--- ⚙️ Preparing Data for TimeGAN ---
   Found 224 time series groups.
   Maximum series length is 29.
   Data prepared with shape: (224, 29, 1)

--- 🏗️ Building TimeGAN Components ---
--- 🛠️ Compiling Models ---
   All models compiled successfully.

--- 🏋️ Training TimeGAN for 5000 epochs ---
   Phase 1/3: Training Autoencoder...
   Autoencoder training complete.

   Phases 2 & 3: Jointly Training Generator and Discriminator...
   Epoch 0, Disc Loss: 0.6948, Gen Loss: 0.6974
   Epoch 100, Disc Loss: 0.7302, Gen Loss: 0.6880
   Epoch 200, Disc Loss: 0.4737, Gen Loss: 1.4834
   Epoch 300, Disc Loss: 0.3598, Gen Loss: 1.7834
   Epoch 400, Disc Loss: 0.3165, Gen Loss: 1.9030
   Epoch 500, Disc Loss: 0.2638, Gen Loss: 2.1191
   Epoch 600, Disc Loss: 0.2250, Gen Loss: 2.4371
   Epoch 700, Disc Loss: 0.1972, Gen Loss: 2.7328
   Epoch 800, Disc Loss: 0.1762, Gen Loss: 2.9885
   Epoch 900, Disc Loss: 0.1597, Gen Loss: 3.2012
   Epoch 1000, Disc Loss: 0.1521, Gen Loss: 3.1832
   Epoch 1100, Dis

: 